# Spark + Kafka MASTER (versi Performa): Atlas Iklim Kota-kota Indonesia
**Big Data & Predictive Analytics** · dijalankan di **PC Master** (1 Master + 5 Worker per kelompok)

**Bagian A (Sel 1-11) kasus nyata:** Master mengalokasikan tugas lewat Kafka, Worker mengunduh data cuaca dan mengolahnya (DataFrame, SQL, EDA), Master menggabungkan, mengolah dengan Spark SQL, dan memvisualisasikan. **Baru:** Sel 7 mengukur **latency tiap tahap** dan **resource** (CPU, RAM, jaringan, disk).

**Bagian B (Sel 12-15) benchmark:** mengukur pengaruh **jumlah partisi (P)** dan **jumlah Worker (W)** terhadap latency, throughput, speedup, efisiensi, keseimbangan beban, dan resource, plus mikrobenchmark latency Kafka.

**Sel 16** menyusun laporan HTML (termasuk hasil performa). **Sel 17** untuk kirim ulang tugas.

**Sebelum mulai:** broker Kafka berjalan, `advertised.listeners` = `MASTER_IP`, port 9092 terbuka, `pip install psutil` di semua PC. Disarankan `group.initial.rebalance.delay.ms=3000` di `server.properties` agar benchmark tidak menunggu terlalu lama.

## Sel 1. Konfigurasi, tabel referensi kota, parameter benchmark, dan topic

In [ ]:
# ===== SEL 1: Konfigurasi, tabel referensi kota, dan topic =====
# !pip install confluent-kafka pyspark pandas matplotlib seaborn psutil      # jalankan sekali jika belum terpasang
import os, sys, io, json, time, uuid, socket, base64, html, threading
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
from confluent_kafka import Producer, Consumer, KafkaError
from confluent_kafka.admin import AdminClient, NewTopic, NewPartitions

# ---------------- ISI SESUAI KELOMPOK ANDA ----------------
KELOMPOK        = "k1"             # k1 ... k8
NOMOR_KELOMPOK  = 1                # 1 ... 8  (menentukan 20 kota milik kelompok ini)
MASTER_IP       = "10.73.3.90"     # IP PC Master ini (HARUS sama dengan advertised.listeners Kafka)
N_WORKER        = 5                # jumlah PC mahasiswa dalam kelompok
PAKAI_CLUSTER_SPARK = False        # True = pengolahan akhir di Spark standalone spark://MASTER_IP:7077
# -----------------------------------------------------------

BOOTSTRAP    = f"{MASTER_IP}:9092"
TOPIC_TUGAS  = f"{KELOMPOK}-tugas"        # Master -> Worker  (kota yang harus diunduh)
TOPIC_HASIL  = f"{KELOMPOK}-hasil"        # Worker -> Master  (ringkasan hasil olahan)
TOPIC_STATUS = f"{KELOMPOK}-status"       # Worker -> Master  (progres & kesiapan)
TOPIC_KONTROL = f"{KELOMPOK}-kontrol"     # Master -> Worker  (perintah benchmark)
TOPIC_BENCH   = f"{KELOMPOK}-bench"       # Worker -> Master  (hasil benchmark)
TOPIC_MIKRO   = f"{KELOMPOK}-mikro"       # topic uji latency Kafka
JUMLAH_KOTA  = 20
JUMLAH_TUGAS = JUMLAH_KOTA + N_WORKER     # 20 tugas kota + 5 tugas percobaan performa (1 per Worker)
MULAI, AKHIR = "2022-01-01", "2024-12-31" # periode data cuaca per jam
RUN_ID       = uuid.uuid4().hex[:8]

# Parameter benchmark partisi x worker (Sel 12-15)
BENCH_W      = [1, 2, 3, 5]       # jumlah Worker yang ikut
BENCH_P      = [1, 5, 10]         # jumlah partisi topic tugas
BENCH_N      = 40                 # jumlah tugas per percobaan
BENCH_BEBAN  = 5                  # berat tiap tugas (1 satuan = 1 juta iterasi CPU, sekitar 0,1 dtk di PC cepat)
BENCH_ULANG  = 1                  # ulangi tiap kombinasi (naikkan ke 2-3 untuk hasil lebih stabil)
HAPUS_TOPIK_BENCH = os.name != "nt"   # Kafka di Windows bisa MATI saat menghapus topic -> di Windows topic percobaan dibiarkan
LAPORAN_DIR  = Path("laporan") / KELOMPOK
LAPORAN_DIR.mkdir(parents=True, exist_ok=True)

# Tabel referensi (dimension table): kota, provinsi, wilayah, zona waktu, lintang, bujur
KOTA_REF = [
    ("Banda Aceh","Aceh","Sumatera","WIB",5.5483,95.3238), ("Medan","Sumatera Utara","Sumatera","WIB",3.5952,98.6722),
    ("Padang","Sumatera Barat","Sumatera","WIB",-0.9471,100.4172), ("Pekanbaru","Riau","Sumatera","WIB",0.5071,101.4478),
    ("Jambi","Jambi","Sumatera","WIB",-1.6101,103.6131), ("Palembang","Sumatera Selatan","Sumatera","WIB",-2.9761,104.7754),
    ("Bengkulu","Bengkulu","Sumatera","WIB",-3.8004,102.2655), ("Bandar Lampung","Lampung","Sumatera","WIB",-5.4292,105.2610),
    ("Pangkalpinang","Kep. Bangka Belitung","Sumatera","WIB",-2.1316,106.1169), ("Batam","Kepulauan Riau","Sumatera","WIB",1.0456,104.0305),
    ("Jakarta","DKI Jakarta","Jawa","WIB",-6.2088,106.8456), ("Serang","Banten","Jawa","WIB",-6.1201,106.1503),
    ("Bandung","Jawa Barat","Jawa","WIB",-6.9175,107.6191), ("Cirebon","Jawa Barat","Jawa","WIB",-6.7320,108.5523),
    ("Semarang","Jawa Tengah","Jawa","WIB",-6.9667,110.4167), ("Surakarta","Jawa Tengah","Jawa","WIB",-7.5755,110.8243),
    ("Yogyakarta","DI Yogyakarta","Jawa","WIB",-7.7956,110.3695), ("Surabaya","Jawa Timur","Jawa","WIB",-7.2575,112.7521),
    ("Malang","Jawa Timur","Jawa","WIB",-7.9666,112.6326), ("Denpasar","Bali","Bali-Nusa Tenggara","WITA",-8.6705,115.2126),
    ("Mataram","Nusa Tenggara Barat","Bali-Nusa Tenggara","WITA",-8.5833,116.1167), ("Kupang","Nusa Tenggara Timur","Bali-Nusa Tenggara","WITA",-10.1772,123.6070),
    ("Pontianak","Kalimantan Barat","Kalimantan","WIB",-0.0263,109.3425), ("Palangka Raya","Kalimantan Tengah","Kalimantan","WIB",-2.2161,113.9135),
    ("Banjarmasin","Kalimantan Selatan","Kalimantan","WITA",-3.3186,114.5944), ("Samarinda","Kalimantan Timur","Kalimantan","WITA",-0.4948,117.1436),
    ("Balikpapan","Kalimantan Timur","Kalimantan","WITA",-1.2379,116.8529), ("Tarakan","Kalimantan Utara","Kalimantan","WITA",3.3000,117.6333),
    ("Manado","Sulawesi Utara","Sulawesi","WITA",1.4748,124.8421), ("Gorontalo","Gorontalo","Sulawesi","WITA",0.5435,123.0568),
    ("Palu","Sulawesi Tengah","Sulawesi","WITA",-0.8917,119.8707), ("Mamuju","Sulawesi Barat","Sulawesi","WITA",-2.6748,118.8884),
    ("Makassar","Sulawesi Selatan","Sulawesi","WITA",-5.1477,119.4327), ("Kendari","Sulawesi Tenggara","Sulawesi","WITA",-3.9985,122.5130),
    ("Ambon","Maluku","Maluku-Papua","WIT",-3.6954,128.1814), ("Ternate","Maluku Utara","Maluku-Papua","WIT",0.7833,127.3667),
    ("Manokwari","Papua Barat","Maluku-Papua","WIT",-0.8615,134.0620), ("Sorong","Papua Barat Daya","Maluku-Papua","WIT",-0.8762,131.2558),
    ("Jayapura","Papua","Maluku-Papua","WIT",-2.5337,140.7181), ("Merauke","Papua Selatan","Maluku-Papua","WIT",-8.4932,140.4018),
]
REF_KOLOM = ["kota", "provinsi", "wilayah", "zona", "lat", "lon"]
# 20 kota per kelompok: loncat 2 kota agar tiap kelompok mencakup banyak wilayah, offset beda per kelompok
KOTA_KELOMPOK = [KOTA_REF[(NOMOR_KELOMPOK * 3 + i * 2) % len(KOTA_REF)] for i in range(JUMLAH_KOTA)]

KOLOM_BULANAN = ["kota", "tahun", "bulan", "suhu_rata", "suhu_maks", "suhu_min", "hujan_total",
                 "hari_hujan", "kelembapan_rata", "angin_rata", "jumlah_hari"]

# Kueri SQL akhir (dijalankan Master pada gabungan semua Worker; view: bulanan, kota_ref)
SQL_FINAL = {
"peringkat_hujan": """
WITH tahunan AS (
    SELECT kota, tahun, SUM(hujan_total) AS hujan_tahun, AVG(suhu_rata) AS suhu_tahun
    FROM bulanan GROUP BY kota, tahun)
SELECT kota,
       ROUND(AVG(hujan_tahun), 0) AS hujan_per_tahun,
       ROUND(AVG(suhu_tahun), 2)  AS suhu_rata,
       RANK() OVER (ORDER BY AVG(hujan_tahun) DESC) AS peringkat_hujan
FROM tahunan GROUP BY kota ORDER BY peringkat_hujan""",
"musiman_wilayah": """
SELECT r.wilayah, b.bulan,
       ROUND(AVG(b.hujan_total), 1) AS hujan_bulanan,
       ROUND(AVG(b.suhu_rata), 2)   AS suhu_rata
FROM bulanan b JOIN kota_ref r ON b.kota = r.kota
GROUP BY r.wilayah, b.bulan ORDER BY r.wilayah, b.bulan""",
"bulan_terbasah": """
SELECT t.kota, t.bulan, ROUND(t.hujan_rata, 1) AS hujan_rata
FROM (SELECT kota, bulan, AVG(hujan_total) AS hujan_rata,
             ROW_NUMBER() OVER (PARTITION BY kota ORDER BY AVG(hujan_total) DESC) AS rn
      FROM bulanan GROUP BY kota, bulan) t
WHERE t.rn = 1 ORDER BY t.kota""",
"anomali_hujan": """
SELECT b.kota, b.tahun, b.bulan, b.hujan_total, ROUND(k.rata, 1) AS rata_kota,
       ROUND(b.hujan_total / k.rata, 2) AS kali_rata
FROM bulanan b
JOIN (SELECT kota, AVG(hujan_total) AS rata FROM bulanan GROUP BY kota) k ON b.kota = k.kota
WHERE b.hujan_total > 2 * k.rata
ORDER BY kali_rata DESC LIMIT 15""",
"ringkas_wilayah": """
SELECT r.wilayah, COUNT(DISTINCT b.kota) AS jumlah_kota,
       ROUND(AVG(b.suhu_rata), 2) AS suhu_rata, ROUND(AVG(b.hujan_total), 1) AS hujan_bulanan,
       ROUND(AVG(b.kelembapan_rata), 1) AS kelembapan
FROM bulanan b JOIN kota_ref r ON b.kota = r.kota
GROUP BY r.wilayah ORDER BY hujan_bulanan DESC""",
}


try:
    import psutil
except ImportError:
    psutil = None          # pip install psutil  (tanpa psutil, pengukuran resource dilewati)

class PemantauResource:
    """Mencuplik CPU, RAM, jaringan, dan disk di thread terpisah selama sebuah blok pekerjaan.
    Pakai: pm = PemantauResource().mulai(); ...pekerjaan...; ringkasan = pm.berhenti()"""
    def __init__(self, interval=0.5):
        self.interval = interval
        self._henti = threading.Event()
        self._t = None
        self.sampel = []

    def _pohon(self):
        p = psutil.Process()
        try:
            return [p] + p.children(recursive=True)        # termasuk proses Java milik PySpark
        except psutil.Error:
            return [p]

    def _cpu_detik(self):
        total = 0.0
        for pr in self._pohon():
            try:
                c = pr.cpu_times(); total += c.user + c.system
            except psutil.Error:
                pass
        return total

    def _rss_mb(self):
        total = 0
        for pr in self._pohon():
            try:
                total += pr.memory_info().rss
            except psutil.Error:
                pass
        return total / 1e6

    def mulai(self):
        if psutil is None:
            return self
        self._t0 = time.perf_counter()
        self._cpu0 = self._cpu_detik()
        self._net0 = psutil.net_io_counters()
        try:
            self._disk0 = psutil.disk_io_counters()
        except Exception:
            self._disk0 = None
        psutil.cpu_percent(None)                            # inisialisasi pembanding
        self._henti.clear(); self.sampel = []
        self._t = threading.Thread(target=self._loop, daemon=True); self._t.start()
        return self

    def _loop(self):
        while not self._henti.wait(self.interval):
            self.sampel.append((psutil.cpu_percent(None), self._rss_mb(), psutil.virtual_memory().percent))

    def berhenti(self):
        if psutil is None:
            return {"psutil": False}
        self._henti.set()
        if self._t:
            self._t.join(2)
        durasi = max(time.perf_counter() - self._t0, 1e-9)
        net1 = psutil.net_io_counters()
        try:
            d1 = psutil.disk_io_counters()
        except Exception:
            d1 = None
        cpu = [s[0] for s in self.sampel] or [psutil.cpu_percent(None)]
        ram = [s[1] for s in self.sampel] or [self._rss_mb()]
        rams = [s[2] for s in self.sampel] or [psutil.virtual_memory().percent]
        cpu_detik = max(0.0, self._cpu_detik() - self._cpu0)
        return {"psutil": True, "durasi": round(durasi, 3),
                "cpu_sistem_rata": round(sum(cpu) / len(cpu), 1), "cpu_sistem_maks": round(max(cpu), 1),
                "cpu_proses_core": round(cpu_detik / durasi, 2),          # rata-rata core yang dipakai proses ini
                "ram_proses_maks_mb": round(max(ram), 1), "ram_sistem_maks_persen": round(max(rams), 1),
                "net_tx_mb": round((net1.bytes_sent - self._net0.bytes_sent) / 1e6, 3),
                "net_rx_mb": round((net1.bytes_recv - self._net0.bytes_recv) / 1e6, 3),
                "disk_tulis_mb": round((d1.write_bytes - self._disk0.write_bytes) / 1e6, 3) if d1 and self._disk0 else None,
                "jumlah_sampel": len(self.sampel)}

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
daftar = [NewTopic(TOPIC_TUGAS,  num_partitions=N_WORKER, replication_factor=1),
          NewTopic(TOPIC_HASIL,  num_partitions=3,        replication_factor=1),
          NewTopic(TOPIC_STATUS, num_partitions=3,        replication_factor=1),
          NewTopic(TOPIC_KONTROL, num_partitions=1,       replication_factor=1),
          NewTopic(TOPIC_BENCH,  num_partitions=3,        replication_factor=1),
          NewTopic(TOPIC_MIKRO,  num_partitions=3,        replication_factor=1)]
for nama, hasil in admin.create_topics(daftar).items():
    try:
        hasil.result(); print("Topic dibuat :", nama)
    except Exception as e:
        print("Dilewati     :", nama, "->", e)

for _ in range(10):                                    # tunggu metadata muncul
    meta = admin.list_topics(timeout=5)
    if TOPIC_TUGAS in meta.topics:
        break
    time.sleep(1)
jml = len(meta.topics[TOPIC_TUGAS].partitions)
if jml < N_WORKER:                                     # topic terlanjur dibuat otomatis dengan partisi kurang
    print(f"PERINGATAN: '{TOPIC_TUGAS}' baru {jml} partisi -> ditambah menjadi {N_WORKER}")
    for _, h in admin.create_partitions([NewPartitions(TOPIC_TUGAS, N_WORKER)]).items():
        h.result()

print(f"\nKelompok {KELOMPOK} | Worker: {N_WORKER} | Kota: {JUMLAH_KOTA} | Tugas total: {JUMLAH_TUGAS} | RUN_ID: {RUN_ID}")
print("Kota kelompok:", ", ".join(k[0] for k in KOTA_KELOMPOK))

## Sel 2. Consumer Master dan monitor
Menyimpan juga perkiraan **selisih jam** tiap Worker (`offset_jam`) dan hasil benchmark.

In [ ]:
# ===== SEL 2: Consumer Master, status, dan monitor =====
from IPython.display import clear_output, display, HTML, FileLink

konsumen = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": f"{KELOMPOK}-master-{RUN_ID}",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": True,
})
konsumen.subscribe([TOPIC_HASIL, TOPIC_STATUS, TOPIC_BENCH])

AKTIF_DETIK = 120
terdaftar, terlihat, status = {}, {}, {}
selesai, gagal = set(), set()
harapan, diterima = {}, Counter()
pelaksana, info_tugas = defaultdict(set), {}
bulanan, perf = {}, {}                 # hasil: kunci unik -> rekaman (duplikat otomatis tertimpa)
NAMA_TUGAS = {}                        # nomor tugas -> kota / "performa" (diisi saat kirim tugas)

# --- pengukuran performa ---
offset_jam = {}                        # worker -> perkiraan (jam Master - jam Worker); min(selisih) = offset + latency terkecil
bench_siap = set()                     # Worker yang siap menerima perintah benchmark
bench_join, bench_tugas, bench_res = {}, {}, {}
mikro = {}                             # (mode, worker, varian) -> hasil mikrobenchmark

def sedot(detik=2.0):
    akhir = time.time() + detik
    while time.time() < akhir:
        msg = konsumen.poll(0.2)
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() != KafkaError._PARTITION_EOF:
                print("Kafka:", msg.error())
            continue
        t_recv = time.time()
        try:
            d = json.loads(msg.value())
        except ValueError:
            continue
        if msg.topic() == TOPIC_STATUS:
            w, st = d.get("worker"), d.get("status")
            if w is None:
                continue
            if "waktu" in d:
                terlihat[w] = max(terlihat.get(w, 0), d["waktu"])
                selisih = t_recv - d["waktu"]
                offset_jam[w] = min(offset_jam.get(w, selisih), selisih)    # perkiraan selisih jam
            if st == "SIAP":
                terdaftar[w] = d.get("partisi", [])
            elif st == "BENCH_SIAP":
                bench_siap.add(w)
            elif st == "BERHENTI":
                terdaftar.pop(w, None); terlihat.pop(w, None); bench_siap.discard(w)
            elif st == "HIDUP":
                pass
            elif d.get("run") == RUN_ID:
                status[w] = d
                t = d["tugas"]
                if st == "SELESAI":
                    d["t_recv"] = t_recv
                    selesai.add(t); gagal.discard(t)
                    harapan[t] = d.get("jumlah", 0); pelaksana[t].add(w); info_tugas[t] = d
                elif st == "GAGAL":
                    gagal.add(t); info_tugas[t] = d
        elif msg.topic() == TOPIC_BENCH:
            if d.get("run") != RUN_ID:
                continue
            j = d.get("jenis")
            if j == "join":
                bench_join[(d["eid"], d["worker"])] = {**d, "t_recv": t_recv}      # terbaru menimpa
            elif j == "tugas":
                d["t_recv"] = t_recv
                bench_tugas[(d["eid"], d["tugas"])] = d
            elif j == "resource":
                bench_res[(d["eid"], d["worker"])] = d
            elif j == "mikro":
                mikro[(d["mode"], d["worker"], d["varian"])] = d
        elif d.get("run") == RUN_ID:
            t = d["tugas"]
            if d.get("jenis") == "bulanan":
                kunci = (t, d["kota"], d["tahun"], d["bulan"])
                if kunci not in bulanan:
                    diterima[t] += 1
                bulanan[kunci] = d
            elif d.get("jenis") == "perf":
                kunci = (t, d["eksperimen"], d["varian"])
                if kunci not in perf:
                    diterima[t] += 1
                perf[kunci] = d

def lengkap():
    """Tugas SELESAI *dan* seluruh rekamannya sudah sampai di Master."""
    return {t for t in selesai if diterima[t] >= harapan.get(t, 0)}

def tampilkan():
    sekarang = time.time()
    aktif = {w for w in terdaftar if sekarang - terlihat.get(w, 0) < AKTIF_DETIK}
    lgk = lengkap()
    print("=" * 90)
    print(f"  MONITOR KELOMPOK {KELOMPOK} - ATLAS IKLIM KOTA-KOTA INDONESIA")
    print("=" * 90)
    print(f"Worker aktif: {len(aktif)}/{N_WORKER} | Tugas selesai: {len(selesai)}/{JUMLAH_TUGAS} | "
          f"Data lengkap: {len(lgk)}/{JUMLAH_TUGAS} | Gagal: {len(gagal)} | "
          f"Rekaman: bulanan={len(bulanan):,} perf={len(perf)}")
    print("\n--- WORKER ---")
    for w in sorted(aktif):
        s = status.get(w)
        if s:
            print(f"{w:<28} partisi {str(terdaftar[w]):<8} tugas {s['tugas']:>2} "
                  f"[{NAMA_TUGAS.get(s['tugas'], '?'):<14}] {s['status']:<10}")
        else:
            print(f"{w:<28} partisi {str(terdaftar[w]):<8} menunggu tugas")
    if gagal:
        print("\nGAGAL:", {t: info_tugas[t].get("galat", "?") for t in sorted(gagal)})

def monitor(refresh=2, otomatis_berhenti=False):
    try:
        while True:
            sedot(refresh)
            clear_output(wait=True)
            tampilkan()
            if otomatis_berhenti and len(lengkap()) + len(gagal) >= JUMLAH_TUGAS:
                if gagal:
                    print(f"\nSELESAI, tetapi {len(gagal)} tugas GAGAL: {sorted(gagal)} -> kirim ulang di Sel 17.")
                else:
                    print("\nSEMUA TUGAS SELESAI DAN DATA LENGKAP.")
                break
            print(f"\nRefresh {refresh} detik | Interrupt (kotak hitam) untuk berhenti")
    except KeyboardInterrupt:
        print("\nMonitor dihentikan.")

print("Consumer Master siap.")

## Sel 3. Tunggu semua Worker siap
Hentikan (Interrupt) ketika **Worker aktif = 5/5**.

In [ ]:
monitor()

## Sel 4. Master mengalokasikan tugas

In [ ]:
# ===== SEL 4: Master mengalokasikan tugas ke Worker =====
produser = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True})
galat_kirim = []
def _hasil_kirim(err, msg):
    if err is not None:
        galat_kirim.append(str(err))

def buat_tugas():
    daftar = []
    for i, (kota, prov, wil, zona, lat, lon) in enumerate(KOTA_KELOMPOK):
        daftar.append({"run": RUN_ID, "tugas": i, "jenis": "kota", "kota": kota, "lat": lat, "lon": lon,
                       "mulai": MULAI, "akhir": AKHIR, "partisi": i % N_WORKER, "dibuat": time.time()})
    for p in range(N_WORKER):               # tugas performa dikirim PALING AKHIR di tiap partisi
        daftar.append({"run": RUN_ID, "tugas": JUMLAH_KOTA + p, "jenis": "performa",
                       "partisi": p, "dibuat": time.time()})
    return daftar

def kirim_tugas(daftar):
    galat_kirim.clear()
    for t in daftar:
        NAMA_TUGAS[t["tugas"]] = t.get("kota", "performa")
        produser.produce(TOPIC_TUGAS, key=str(t["tugas"]), value=json.dumps(t).encode("utf-8"),
                         partition=t["partisi"], on_delivery=_hasil_kirim)
    sisa = produser.flush(30)
    if sisa or galat_kirim:
        print(f"PERINGATAN: belum terkirim={sisa}, galat={galat_kirim[:2]}")
    else:
        print(f"{len(daftar)} tugas terkirim & dikonfirmasi broker ke '{TOPIC_TUGAS}'")

SEMUA_TUGAS = buat_tugas()
kirim_tugas(SEMUA_TUGAS)
tabel = pd.DataFrame([{"tugas": t["tugas"], "jenis": t["jenis"], "kota": t.get("kota", "-"), "partisi": t["partisi"]}
                      for t in SEMUA_TUGAS])
print("\nAlokasi tugas ke partisi (1 partisi = 1 Worker):")
print(tabel.groupby("partisi")["kota"].apply(lambda s: ", ".join(s)).to_string())

## Sel 5. Pantau sampai semua tugas selesai

In [ ]:
monitor(otomatis_berhenti=True)

## Sel 6. Bukti pembagian tugas

In [ ]:
# ===== SEL 6: Bukti pembagian tugas (siapa mengerjakan apa) =====
print(f"{'TUGAS':<6}{'JENIS':<10}{'KOTA':<16}{'DIKERJAKAN OLEH':<26}{'REKAMAN':>8}{'BARIS':>10}{'UNDUH(s)':>10}{'PROSES(s)':>10}")
print("-" * 98)
for t in range(JUMLAH_TUGAS):
    w = ", ".join(sorted(pelaksana[t])) or "-"
    inf = info_tugas.get(t, {})
    jenis = "kota" if t < JUMLAH_KOTA else "performa"
    print(f"{t:<6}{jenis:<10}{NAMA_TUGAS.get(t, '-'):<16}{w:<26}{diterima[t]:>8}"
          f"{inf.get('baris', 0):>10,}{inf.get('detik_unduh', 0):>10}{inf.get('detik_proses', 0):>10}")
per_worker = Counter(w for t in range(JUMLAH_TUGAS) for w in pelaksana[t])
print(f"\nWorker yang bekerja : {len(per_worker)} dari {N_WORKER}")
print(f"Tugas per worker    : {dict(per_worker)}")
print(f"Tugas tanpa pelaksana: {[t for t in range(JUMLAH_TUGAS) if not pelaksana[t]]}")
print(f"Tugas dikerjakan >1 worker (dobel, sudah digabung aman): {[t for t in range(JUMLAH_TUGAS) if len(pelaksana[t]) > 1]}")

## Sel 7. Latency dan resource pada kasus nyata (BARU)
Menguraikan waktu tiap tugas menjadi **antre, unduh, proses, kirim, pengiriman hasil**, serta resource tiap Worker.

In [ ]:
# ===== SEL 7: Latency dan resource pada kasus nyata (20 kota) =====
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def tabel_latency(kota_saja=True):
    """Uraikan waktu tiap tugas. Semua dalam jam Master (jam Worker dikoreksi dengan offset_jam)."""
    baris = []
    for t, d in sorted(info_tugas.items()):
        if kota_saja and t >= JUMLAH_KOTA or d.get("status") != "SELESAI" or "t_terima" not in d:
            continue
        off = offset_jam.get(d["worker"], 0.0)
        res = d.get("res", {}) or {}
        baris.append({
            "tugas": t, "kota": NAMA_TUGAS.get(t, "?"), "worker": d["worker"],
            "antri": d["t_terima"] + off - d["t_dibuat"],            # menunggu di Kafka + antre di belakang tugas lain
            "unduh": d.get("detik_unduh", 0), "proses": d.get("detik_proses", 0), "kirim": d.get("detik_kirim", 0),
            "layanan": d["t_selesai"] - d["t_mulai"],
            "pengiriman": d["t_recv"] - (d["t_selesai"] + off),      # SELESAI -> sampai di Master
            "end_to_end": d["t_recv"] - d["t_dibuat"],
            "baris": d.get("baris", 0),
            "cpu_core": res.get("cpu_proses_core"), "cpu_sistem": res.get("cpu_sistem_rata"),
            "ram_mb": res.get("ram_proses_maks_mb"), "net_rx_mb": res.get("net_rx_mb"),
            "net_tx_mb": res.get("net_tx_mb"), "disk_mb": res.get("disk_tulis_mb")})
    return pd.DataFrame(baris)

lat = tabel_latency()
if lat.empty:
    print("Belum ada data. Jalankan Sel 4-5 sampai semua tugas selesai.")
else:
    persen = lambda s, q: s.quantile(q / 100)
    print(f"Tugas kota terukur : {len(lat)} | selisih jam Master-Worker (detik): "
          f"{ {w: round(v, 3) for w, v in sorted(offset_jam.items())} }")
    ring = pd.DataFrame({"rata": lat[["antri", "unduh", "proses", "kirim", "pengiriman", "end_to_end"]].mean(),
                         "p50": lat[["antri", "unduh", "proses", "kirim", "pengiriman", "end_to_end"]].median(),
                         "p95": lat[["antri", "unduh", "proses", "kirim", "pengiriman", "end_to_end"]].quantile(0.95),
                         "maks": lat[["antri", "unduh", "proses", "kirim", "pengiriman", "end_to_end"]].max()}).round(2)
    print("\nLatency per tahap (detik):"); display(ring)

    # 1. uraian latency per tugas
    urut = lat.sort_values("end_to_end")
    fig, ax = plt.subplots(1, 2, figsize=(14, 0.3 * len(urut) + 2.5))
    kiri = urut.set_index("kota")[["antri", "unduh", "proses", "kirim", "pengiriman"]]
    kiri.plot.barh(stacked=True, ax=ax[0], color=["#CBD5E0", "#F6AD55", "#63B3ED", "#68D391", "#B794F4"])
    ax[0].set(title="Uraian latency end-to-end per kota (detik)", xlabel="detik", ylabel="")
    sns.boxplot(data=lat.melt(value_vars=["antri", "unduh", "proses", "kirim", "pengiriman"], var_name="tahap", value_name="detik"),
                x="tahap", y="detik", ax=ax[1])
    ax[1].set(title="Sebaran waktu tiap tahap", xlabel="", ylabel="detik")
    fig.tight_layout(); fig.savefig(LAPORAN_DIR / "7a_latency_kasus.png", dpi=110); plt.show(); plt.close(fig)

    # 2. resource per Worker
    if lat["cpu_core"].notna().any():
        rw = lat.groupby("worker").agg(cpu_core=("cpu_core", "mean"), cpu_sistem=("cpu_sistem", "mean"),
                                       ram_mb=("ram_mb", "max"), net_rx_mb=("net_rx_mb", "sum"),
                                       net_tx_mb=("net_tx_mb", "sum"), disk_mb=("disk_mb", "sum")).round(2).reset_index()
        print("\nResource per Worker selama mengerjakan tugas kota:"); display(rw)
        fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
        for a, (k, jdl) in zip(ax, [("cpu_sistem", "CPU sistem rata-rata (%)"), ("ram_mb", "RAM proses maks (MB)"),
                                    ("net_rx_mb", "Jaringan diterima (MB)"), ("disk_mb", "Disk tulis (MB)")]):
            sns.barplot(data=rw, x="worker", y=k, ax=a, color="#4C78A8"); a.set(title=jdl, xlabel="", ylabel="")
            plt.setp(a.get_xticklabels(), rotation=30, ha="right")
        fig.tight_layout(); fig.savefig(LAPORAN_DIR / "7b_resource_kasus.png", dpi=110); plt.show(); plt.close(fig)
    else:
        print("\n(psutil tidak terpasang di Worker: resource tidak terukur. pip install psutil)")
    gambar_kasus = {"7a_latency_kasus": LAPORAN_DIR / "7a_latency_kasus.png"}

## Sel 8. Spark Master: sesi dan operasi cluster

In [ ]:
# ===== SEL 8: Spark Master - sesi untuk pengolahan akhir & operasi cluster =====
from pyspark.sql import SparkSession, functions as F

IP = MASTER_IP                      # notebook ini berjalan di PC Master

builder = (SparkSession.builder.appName(f"{KELOMPOK}-final-report")
           .config("spark.driver.host", IP).config("spark.driver.bindAddress", IP)
           .config("spark.pyspark.python", sys.executable)
           .config("spark.pyspark.driver.python", sys.executable))
if PAKAI_CLUSTER_SPARK:
    builder = (builder.master(f"spark://{MASTER_IP}:7077")
               .config("spark.cores.max", str(2 * N_WORKER))       # batasi pemakaian core cluster
               .config("spark.executor.memory", "1g"))
else:
    builder = builder.master("local[*]")
spark = builder.getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("=== OPERASI CLUSTER ===")
print("Versi Spark         :", spark.version)
print("Master              :", sc.master)
print("Spark UI            :", sc.uiWebUrl)
print("defaultParallelism  :", sc.defaultParallelism, "(jumlah core yang tersedia)")
print("shuffle.partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE aktif           :", spark.conf.get("spark.sql.adaptive.enabled", "true"))
try:
    print("Executor (termasuk driver):", sc._jsc.sc().getExecutorMemoryStatus().size())
except Exception:
    pass
host = sc.parallelize(range(20), 20).map(lambda _: socket.gethostname()).distinct().collect()
print("Task 20 partisi berjalan di host:", sorted(host))
print("\nBuka Spark UI di browser: tab Jobs, Stages, Executors, SQL.")

## Sel 9. Pengolahan akhir: DataFrame + Spark SQL

In [ ]:
# ===== SEL 9: Pengolahan akhir - gabungan semua Worker (DataFrame + SQL) =====
if not bulanan:
    raise RuntimeError("Belum ada data bulanan dari Worker. Jalankan Sel 5 sampai muncul 'SEMUA TUGAS SELESAI DAN DATA LENGKAP'.")
pdf_bulanan = pd.DataFrame([{k: d[k] for k in KOLOM_BULANAN} for d in bulanan.values()])
print(f"Gabungan data bulanan: {len(pdf_bulanan):,} baris dari {pdf_bulanan['kota'].nunique()} kota "
      f"(seharusnya {JUMLAH_KOTA * 36})")

bulanan_sdf = spark.createDataFrame(pdf_bulanan)
bulanan_sdf.createOrReplaceTempView("bulanan")
ref_sdf = spark.createDataFrame(KOTA_REF, REF_KOLOM)
ref_sdf.createOrReplaceTempView("kota_ref")

hasil_sql = {}
for nama, q in SQL_FINAL.items():
    hasil_sql[nama] = spark.sql(q).toPandas()
    print(f"\n--- {nama} ---")
    display(hasil_sql[nama].head(12))

# Versi DataFrame API untuk ringkasan wilayah + rencana eksekusi (lihat BroadcastHashJoin)
ringkas_df = (bulanan_sdf.join(F.broadcast(ref_sdf), "kota")
              .groupBy("wilayah").agg(F.round(F.avg("suhu_rata"), 2).alias("suhu_rata"),
                                      F.round(F.avg("hujan_total"), 1).alias("hujan_bulanan")))
print("\nRencana eksekusi join (DataFrame API):")
ringkas_df.explain()

## Sel 10. Visualisasi akhir (matplotlib + seaborn)

In [ ]:
# ===== SEL 10: Visualisasi akhir (matplotlib + seaborn) =====
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
KW_SD = {"errorbar": "sd"} if tuple(int(x) for x in sns.__version__.split(".")[:2]) >= (0, 12) else {"ci": "sd"}
NAMA_BULAN = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]
gambar = {}

def simpan(fig, nama):
    p = LAPORAN_DIR / f"{nama}.png"
    fig.tight_layout(); fig.savefig(p, dpi=110); gambar[nama] = p
    plt.show(); plt.close(fig)

def gambar_final(pdf, hasil):
    ref = pd.DataFrame(KOTA_REF, columns=REF_KOLOM)
    pdf = pdf.merge(ref[["kota", "wilayah", "zona"]], on="kota")
    # 1. heatmap kota x bulan (hujan)
    urut = hasil["peringkat_hujan"]["kota"].tolist()
    pivot = pdf.groupby(["kota", "bulan"])["hujan_total"].mean().unstack().reindex(urut)
    fig, ax = plt.subplots(figsize=(11, 0.38 * len(urut) + 2))
    sns.heatmap(pivot, cmap="YlGnBu", annot=True, fmt=".0f", ax=ax, xticklabels=NAMA_BULAN,
                cbar_kws={"label": "mm / bulan"})
    ax.set(title="Curah hujan rata-rata per bulan (mm): kota diurutkan dari terbasah", xlabel="", ylabel="")
    simpan(fig, "1_heatmap_hujan")
    # 2. peringkat hujan tahunan
    pr = hasil["peringkat_hujan"].merge(ref[["kota", "wilayah"]], on="kota")
    fig, ax = plt.subplots(figsize=(10, 0.35 * len(pr) + 1.5))
    sns.barplot(data=pr, y="kota", x="hujan_per_tahun", hue="wilayah", dodge=False, ax=ax)
    ax.set(title="Curah hujan per tahun (mm)", xlabel="mm / tahun", ylabel="")
    simpan(fig, "2_peringkat_hujan")
    # 3. suhu per wilayah
    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.boxplot(data=pdf, x="wilayah", y="suhu_rata", ax=ax)
    ax.set(title="Sebaran suhu rata-rata bulanan per wilayah", xlabel="", ylabel="\u00b0C")
    plt.setp(ax.get_xticklabels(), rotation=20)
    simpan(fig, "3_suhu_wilayah")
    # 4. pola musiman
    ms = hasil["musiman_wilayah"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.lineplot(data=ms, x="bulan", y="hujan_bulanan", hue="wilayah", marker="o", ax=ax[0])
    sns.lineplot(data=ms, x="bulan", y="suhu_rata", hue="wilayah", marker="o", ax=ax[1], legend=False)
    ax[0].set(title="Pola musim hujan per wilayah", ylabel="mm / bulan"); ax[1].set(title="Pola suhu per wilayah", ylabel="\u00b0C")
    for a in ax: a.set_xticks(range(1, 13)); a.set_xticklabels(NAMA_BULAN)
    simpan(fig, "4_musiman")
    # 5. kelembapan vs hujan
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=pdf, x="kelembapan_rata", y="hujan_total", hue="wilayah", alpha=0.7, ax=ax)
    ax.set(title="Kelembapan vs curah hujan bulanan", xlabel="kelembapan rata-rata (%)", ylabel="hujan (mm)")
    simpan(fig, "5_kelembapan_hujan")
    return pdf

pdf_final = gambar_final(pdf_bulanan, hasil_sql)

## Sel 11. Laporan percobaan performa Spark (E1-E5) dari semua Worker

In [ ]:
# ===== SEL 11: Laporan percobaan performa Spark dari semua Worker =====
KOLOM_PERF = ["worker", "eksperimen", "varian", "detik", "catatan"]
pdf_perf = pd.DataFrame([{k: d[k] for k in KOLOM_PERF} for d in perf.values()], columns=KOLOM_PERF)   # columns= : tetap ada walau kosong
pdf_perf = pdf_perf[~pdf_perf["varian"].str.contains("dilewati")]
if pdf_perf.empty:
    print("Belum ada data performa.")
else:
    ringkas = (pdf_perf.groupby(["eksperimen", "varian"], sort=False)["detik"]
               .agg(rata="mean", sd="std", n="count").round(3).reset_index())
    display(ringkas)
    eks = list(ringkas["eksperimen"].unique())
    fig, axs = plt.subplots(1, len(eks), figsize=(3.6 * len(eks), 4), squeeze=False)
    for ax, e in zip(axs[0], eks):
        sub = pdf_perf[pdf_perf["eksperimen"] == e]
        sns.barplot(data=sub, x="varian", y="detik", ax=ax, **KW_SD)
        ax.set_title(e, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("detik" if e == eks[0] else "")
        plt.setp(ax.get_xticklabels(), rotation=25)
    fig.suptitle(f"Waktu eksekusi (rata-rata {pdf_perf['worker'].nunique()} Worker, batang galat = sd)")
    simpan(fig, "6_performa")

## Sel 12. Benchmark: fungsi bantu dan kesiapan Worker (BARU)
Pastikan semua Worker sudah menjalankan **Sel 8 Worker** (loop utama).

In [ ]:
# ===== SEL 12: Benchmark - fungsi bantu dan kesiapan Worker =====
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def tunggu(kondisi, timeout=60.0, jeda=0.2):
    """Baca Kafka sambil menunggu kondisi terpenuhi. True jika terpenuhi sebelum timeout."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        sedot(jeda)
        if kondisi():
            return True
    return False

def kirim_kontrol(obj):
    obj = {"run": RUN_ID, **obj}
    produser.produce(TOPIC_KONTROL, value=json.dumps(obj).encode("utf-8"), partition=0)
    produser.flush(10)

def buat_topik(nama, partisi):
    for _, f in admin.create_topics([NewTopic(nama, num_partitions=partisi, replication_factor=1)]).items():
        try:
            f.result()
        except Exception as e:
            print("  topic:", e)
    for _ in range(20):                                    # tunggu metadata muncul
        if nama in admin.list_topics(timeout=5).topics:
            return
        time.sleep(0.5)

def hapus_topik(nama):
    if not HAPUS_TOPIK_BENCH:
        return
    try:
        for _, f in admin.delete_topics([nama]).items():
            f.result()
    except Exception:
        pass

print("Menunggu Worker siap menerima perintah benchmark (Sel 8 Worker harus berjalan) ...")
ok = tunggu(lambda: len(bench_siap) >= N_WORKER, timeout=60)
print(f"Worker siap benchmark: {len(bench_siap)}/{N_WORKER}", "-> OK" if ok else "-> belum semua; lanjutkan hanya jika sengaja memakai lebih sedikit Worker")
print("Selisih jam Master-Worker (detik, perkiraan):", {w: round(v, 3) for w, v in sorted(offset_jam.items())})

## Sel 13. Mikrobenchmark Kafka (BARU)
Latency produce→consume untuk 3 konfigurasi (`acks`, `linger.ms`) dan throughput dengan/tanpa kompresi, dijalankan **bergiliran** lalu **serempak**.

In [ ]:
# ===== SEL 13: Mikrobenchmark Kafka - latency produce->consume dan throughput =====
def jalankan_mikro(mode):
    if mode == "bergiliran":                               # satu Worker pada satu waktu (tanpa persaingan)
        for n in range(1, N_WORKER + 1):
            kirim_kontrol({"aksi": "MIKRO", "eid": f"m-{mode}-{n}", "topik": TOPIC_MIKRO, "untuk": n,
                           "mode": mode, "N_lat": 100, "N_thr": 2000})
            tunggu(lambda: sum(1 for k in mikro if k[0] == mode and k[1].endswith(f"-w{n}")) >= 5, timeout=120)
    else:                                                  # semua Worker sekaligus (persaingan jaringan & broker)
        kirim_kontrol({"aksi": "MIKRO", "eid": f"m-{mode}", "topik": TOPIC_MIKRO, "untuk": 0,
                       "mode": mode, "N_lat": 100, "N_thr": 2000})
        tunggu(lambda: sum(1 for k in mikro if k[0] == mode) >= 5 * N_WORKER, timeout=180)

mikro.clear()                                          # sel ini boleh dijalankan ulang: buang hasil lama
pm_master = PemantauResource().mulai()
jalankan_mikro("bergiliran")
jalankan_mikro("serempak")
res_mikro_master = pm_master.berhenti()

df_mikro = pd.DataFrame([{k: d[k] for k in ("mode", "worker", "varian", "p50_ms", "p95_ms", "p99_ms", "rata_ms", "throughput")}
                         for d in mikro.values()])
if df_mikro.empty:
    print("Belum ada data mikrobenchmark.")
else:
    lat_df = df_mikro[df_mikro["varian"].str.startswith("L")]
    thr_df = df_mikro[df_mikro["varian"].str.startswith("T")]
    ringkas_mikro = (lat_df.groupby(["mode", "varian"])[["p50_ms", "p95_ms", "p99_ms"]].mean().round(2).reset_index())
    print("Latency produce->consume (ms), rata-rata antar Worker:"); display(ringkas_mikro)
    print("Throughput produser (pesan/detik):"); display(thr_df.groupby(["mode", "varian"])["throughput"].mean().round(1).reset_index())
    print("Resource Master (broker) selama mikrobenchmark:", res_mikro_master)

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
    for a, mode in zip(ax[:2], ["bergiliran", "serempak"]):
        sub = lat_df[lat_df["mode"] == mode].melt(id_vars=["worker", "varian"], value_vars=["p50_ms", "p95_ms", "p99_ms"],
                                                  var_name="persentil", value_name="ms")
        if not sub.empty:
            sns.barplot(data=sub, x="varian", y="ms", hue="persentil", ax=a)
        a.set(title=f"Latency produce\u2192consume ({mode})", xlabel="", ylabel="milidetik")
        plt.setp(a.get_xticklabels(), rotation=15, ha="right")
    if not thr_df.empty:
        sns.barplot(data=thr_df, x="varian", y="throughput", hue="mode", ax=ax[2])
    ax[2].set(title="Throughput produser (pesan/detik)", xlabel="", ylabel="")
    plt.setp(ax[2].get_xticklabels(), rotation=15, ha="right")
    fig.tight_layout(); fig.savefig(LAPORAN_DIR / "b0_mikro_kafka.png", dpi=110); plt.show(); plt.close(fig)

## Sel 14. Matriks eksperimen partisi × Worker (BARU)
Menjalankan semua kombinasi `BENCH_W` × `BENCH_P` (didahului percobaan pemanasan yang dibuang). Perkiraan waktu: beberapa menit.

In [ ]:
# ===== SEL 14: Matriks eksperimen - jumlah partisi x jumlah Worker =====
bench_meta = {}
SESI = uuid.uuid4().hex[:4]                           # eid unik per eksekusi sel (sel ini boleh dijalankan ulang)

def jalankan_eksperimen(eid, W, P, N, beban, ulang, warmup=False):
    topik = f"{KELOMPOK}-b-{RUN_ID}-{eid}"
    buat_topik(topik, P)
    t_ctl = time.time()
    kirim_kontrol({"aksi": "MULAI", "eid": eid, "topik": topik, "group": topik, "W": W, "P": P, "N": N})

    def join_stabil():                                     # W Worker bergabung & seluruh P partisi terbagi
        j = {w: d for (e, w), d in bench_join.items() if e == eid}
        return len(j) == W and sorted(p for d in j.values() for p in d["partisi"]) == list(range(P))
    if not tunggu(join_stabil, timeout=90):
        print("  PERINGATAN: consumer group belum stabil")
    t_join = time.time() - t_ctl                           # biaya rebalance

    pm = PemantauResource().mulai()
    t0 = time.time()
    for i in range(N):
        k = KOTA_KELOMPOK[i % len(KOTA_KELOMPOK)]
        tg = {"run": RUN_ID, "eid": eid, "tugas": i, "kota": k[0], "lat": k[4], "lon": k[5],
              "beban": beban, "dibuat": time.time()}
        produser.produce(topik, key=str(i), value=json.dumps(tg).encode("utf-8"), partition=i % P)
    produser.flush(30)
    lengkap_ok = tunggu(lambda: sum(1 for (e, _) in bench_tugas if e == eid) >= N, timeout=max(90, N * beban * 3))
    t_akhir = max((d["t_recv"] for (e, _), d in bench_tugas.items() if e == eid), default=time.time())
    res_master = pm.berhenti()
    kirim_kontrol({"aksi": "SELESAI", "eid": eid})
    tunggu(lambda: sum(1 for (e, _) in bench_res if e == eid) >= W, timeout=30)
    hapus_topik(topik)
    bench_meta[eid] = {"eid": eid, "W": W, "P": P, "N": N, "beban": beban, "ulang": ulang, "warmup": warmup,
                       "t_join": round(t_join, 3), "makespan": round(t_akhir - t0, 3),
                       "res_master": res_master, "lengkap": lengkap_ok}
    print(f"  selesai: makespan={bench_meta[eid]['makespan']} dtk, join={bench_meta[eid]['t_join']} dtk, "
          f"throughput={N / bench_meta[eid]['makespan']:.2f} tugas/dtk")

MATRIKS = [(W, P) for P in BENCH_P for W in BENCH_W if W <= N_WORKER]
print(f"Rencana: {len(MATRIKS)} kombinasi x {BENCH_ULANG} ulangan, {BENCH_N} tugas/percobaan, beban {BENCH_BEBAN}.")

print("\n[pemanasan] percobaan kecil dibuang agar JIT/cache/koneksi sudah hangat")
jalankan_eksperimen(f"{SESI}-e00", min(2, N_WORKER), 2, 6, 1, 0, warmup=True)

nomor = 0
for u in range(1, BENCH_ULANG + 1):
    for W, P in MATRIKS:
        nomor += 1
        print(f"\n[{nomor}/{len(MATRIKS) * BENCH_ULANG}] Worker={W}  Partisi={P}  (ulangan {u})")
        jalankan_eksperimen(f"{SESI}-e{nomor:02d}", W, P, BENCH_N, BENCH_BEBAN, u)
print("\nSemua percobaan selesai. Lanjut ke Sel 15 untuk analisis.")

## Sel 15. Analisis benchmark (BARU)
Tabel dan 4 grafik: skalabilitas (throughput, speedup, efisiensi), persentil latency, uraian latency dan keseimbangan beban, resource.

In [ ]:
# ===== SEL 15: Analisis benchmark - throughput, speedup, efisiensi, latency, resource =====
def tabel_tugas_bench():
    """Satu baris per tugas; waktu lintas-PC dikoreksi dengan offset_jam."""
    baris = []
    for (eid, t), d in bench_tugas.items():
        m = bench_meta.get(eid)
        if not m or m.get("warmup"):
            continue
        off = offset_jam.get(d["worker"], 0.0)
        baris.append({"eid": eid, "W": m["W"], "P": m["P"], "ulang": m["ulang"], "tugas": t,
                      "worker": d["worker"], "partisi": d["partisi"],
                      "antri": d["t_terima"] + off - d["t_dibuat"],             # tunggu di Kafka + antre
                      "layanan": d["detik_layanan"],                            # waktu kerja sebenarnya
                      "pengiriman": d["t_recv"] - (d["t_selesai"] + off),       # hasil -> Master
                      "end_to_end": d["t_recv"] - d["t_dibuat"]})
    return pd.DataFrame(baris)

def ringkas_eksperimen(dt):
    baris = []
    for eid, g in dt.groupby("eid"):
        m = bench_meta[eid]; W, P, N = m["W"], m["P"], len(g)
        per_w = g.groupby("worker").size().tolist()
        x = per_w + [0] * (W - len(per_w))                                     # Worker menganggur dihitung 0
        jain = (sum(x) ** 2) / (W * sum(v * v for v in x)) if sum(x) else 0.0  # 1 = merata sempurna
        res_w = [d["res"] for (e, w), d in bench_res.items() if e == eid and d["res"].get("psutil")]
        rata = lambda k: round(sum(r[k] for r in res_w) / len(res_w), 2) if res_w else None
        rm = m["res_master"] if m["res_master"].get("psutil") else {}
        baris.append({"eid": eid, "W": W, "P": P, "ulang": m["ulang"], "N": N,
                      "makespan": m["makespan"], "throughput": N / m["makespan"], "t_join": m["t_join"],
                      "worker_aktif": len(per_w), "jain": round(jain, 3),
                      "utilisasi": g["layanan"].sum() / (W * m["makespan"]),
                      "p50": g["end_to_end"].quantile(.50), "p95": g["end_to_end"].quantile(.95),
                      "p99": g["end_to_end"].quantile(.99), "antri": g["antri"].mean(),
                      "layanan": g["layanan"].mean(), "pengiriman": g["pengiriman"].mean(),
                      "cpu_worker": rata("cpu_sistem_rata"), "core_worker": rata("cpu_proses_core"),
                      "ram_worker_mb": rata("ram_proses_maks_mb"), "net_rx_worker_mb": rata("net_rx_mb"),
                      "cpu_master": rm.get("cpu_sistem_rata"), "net_rx_master_mb": rm.get("net_rx_mb"),
                      "net_tx_master_mb": rm.get("net_tx_mb")})
    return pd.DataFrame(baris)

dt_bench = tabel_tugas_bench()
if dt_bench.empty:
    print("Belum ada hasil benchmark. Jalankan Sel 14 dulu.")
else:
    de = ringkas_eksperimen(dt_bench)
    for k in ("cpu_worker", "core_worker", "ram_worker_mb", "net_rx_worker_mb", "cpu_master", "net_rx_master_mb", "net_tx_master_mb"):
        de[k] = pd.to_numeric(de[k], errors="coerce")           # tanpa psutil isinya None -> jadikan NaN agar kolom tidak hilang
    bench_agg = de.groupby(["P", "W"], as_index=False).mean(numeric_only=True)
    # speedup, efisiensi, fraksi serial (Karp-Flatt) terhadap W terkecil pada P yang sama
    dasar = bench_agg.sort_values("W").groupby("P").first()[["W", "makespan"]].rename(columns={"W": "W0", "makespan": "T0"})
    bench_agg = bench_agg.merge(dasar, on="P")
    bench_agg["speedup"] = bench_agg["T0"] / bench_agg["makespan"] * bench_agg["W0"]
    import math
    bench_agg["efisiensi"] = bench_agg["speedup"] / bench_agg["W"]
    # prediksi teori: makespan sebanding dengan ceil(P/W) (partisi dibagi berurutan ke Worker, tugas sama berat)
    bench_agg["speedup_teori"] = [math.ceil(p / w0) / math.ceil(p / w) for p, w0, w in zip(bench_agg["P"], bench_agg["W0"], bench_agg["W"])]
    bench_agg["serial_kf"] = [((1 / s) - (1 / w)) / (1 - 1 / w) if w > 1 else float("nan")
                              for s, w in zip(bench_agg["speedup"], bench_agg["W"])]
    tampil = bench_agg[["P", "W", "makespan", "throughput", "speedup", "speedup_teori", "efisiensi", "serial_kf", "utilisasi",
                        "worker_aktif", "jain", "t_join", "p50", "p95", "p99"]].round(3)
    print("Ringkasan percobaan (rata-rata antar ulangan):"); display(tampil)
    ringkas_bench = tampil

    P_list, W_list = sorted(bench_agg["P"].unique()), sorted(bench_agg["W"].unique())
    warna = dict(zip(P_list, sns.color_palette("tab10", len(P_list))))
    def garis(ax, kolom, judul, ideal=False, ylabel=""):
        for P in P_list:
            s = bench_agg[bench_agg["P"] == P].sort_values("W")
            ax.plot(s["W"], s[kolom], "-o", color=warna[P], label=f"P={P}")
        if ideal:
            w = [min(W_list), max(W_list)]
            s0 = bench_agg[bench_agg["W"] == min(W_list)][kolom].max()
            ax.plot(w, [s0 * x / min(W_list) for x in w], "k--", lw=1, label="ideal")
        ax.set(title=judul, xlabel="jumlah Worker", ylabel=ylabel, xticks=W_list); ax.legend(fontsize=8)
    gb = {}
    def simpan_b(fig, nama):
        p = LAPORAN_DIR / f"{nama}.png"; fig.tight_layout(); fig.savefig(p, dpi=110); gb[nama] = p; plt.show(); plt.close(fig)

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    garis(ax[0], "throughput", "Throughput (tugas/detik)", ideal=True, ylabel="tugas/detik")
    garis(ax[1], "speedup", "Speedup (garis) vs prediksi teori (x)", ideal=False, ylabel="kali")
    for P in P_list:
        s_ = bench_agg[bench_agg["P"] == P].sort_values("W"); ax[1].plot(s_["W"], s_["speedup_teori"], ":x", color=warna[P], alpha=0.8)
    ax[1].plot([min(W_list), max(W_list)], [min(W_list), max(W_list)], "k--", lw=1)
    piv = bench_agg.pivot(index="P", columns="W", values="efisiensi")
    sns.heatmap(piv, annot=True, fmt=".2f", cmap="YlGn", vmin=0, vmax=1, ax=ax[2]); ax[2].set(title="Efisiensi paralel (speedup / W)")
    simpan_b(fig, "b1_skalabilitas")

    fig, ax = plt.subplots(1, len(P_list), figsize=(4.6 * len(P_list), 4), sharey=True, squeeze=False)
    for a, P in zip(ax[0], P_list):
        s = bench_agg[bench_agg["P"] == P].sort_values("W")
        for k, c in (("p50", "#68D391"), ("p95", "#F6AD55"), ("p99", "#FC8181")):
            a.plot(s["W"], s[k], "-o", color=c, label=k)
        a.set(title=f"Latency end-to-end, P={P}", xlabel="jumlah Worker", ylabel="detik", xticks=W_list); a.legend()
    simpan_b(fig, "b2_latency_persentil")

    urut = bench_agg.sort_values(["P", "W"]); lab = [f"P{p}-W{w}" for p, w in zip(urut["P"], urut["W"])]
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
    bawah = [0] * len(urut)
    for k, c, n in (("antri", "#CBD5E0", "antre di Kafka"), ("layanan", "#63B3ED", "kerja (layanan)"), ("pengiriman", "#B794F4", "kirim hasil")):
        ax[0].bar(lab, urut[k], bottom=bawah, color=c, label=n); bawah = [b + v for b, v in zip(bawah, urut[k])]
    ax[0].set(title="Uraian rata-rata latency per tugas", ylabel="detik"); ax[0].legend(); plt.setp(ax[0].get_xticklabels(), rotation=60, ha="right")
    sns.heatmap(bench_agg.pivot(index="P", columns="W", values="jain"), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax[1])
    ax[1].set(title="Keseimbangan beban (indeks Jain, 1 = merata)")
    simpan_b(fig, "b3_dekomposisi_keseimbangan")

    fig, ax = plt.subplots(1, 4, figsize=(17, 4))
    if bench_agg["cpu_worker"].notna().any():
        garis(ax[0], "cpu_worker", "CPU sistem Worker (%)", ylabel="%")
        garis(ax[1], "core_worker", "Core terpakai oleh proses Worker", ylabel="core")
        garis(ax[2], "cpu_master", "CPU sistem Master/broker (%)", ylabel="%")
        garis(ax[3], "net_rx_master_mb", "Jaringan diterima Master (MB)", ylabel="MB")
    else:
        for a in ax: a.text(0.5, 0.5, "psutil tidak terpasang\n(pip install psutil)", ha="center", transform=a.transAxes)
    simpan_b(fig, "b4_resource")

    # Little's law: rata-rata tugas dalam sistem = throughput x latency rata-rata
    bench_agg["little_L"] = bench_agg["throughput"] * (bench_agg["antri"] + bench_agg["layanan"] + bench_agg["pengiriman"])
    print("\nHukum Little (L = \u03bb \u00d7 waktu rata-rata) - tugas dalam sistem pada rata-rata:")
    display(bench_agg[["P", "W", "throughput", "little_L"]].round(2))
    bench_gambar = gb

## Sel 16. Laporan akhir (HTML)

In [ ]:
# ===== SEL 16: Laporan akhir (HTML) =====
def b64(p):
    return base64.b64encode(Path(p).read_bytes()).decode()

def temuan(pdf, hasil):
    pr = hasil["peringkat_hujan"]
    basah, kering = pr.iloc[0], pr.iloc[-1]
    panas, sejuk = pr.sort_values("suhu_rata").iloc[-1], pr.sort_values("suhu_rata").iloc[0]
    bulan_basah = int(pdf.groupby("bulan")["hujan_total"].mean().idxmax())
    bulan_kering = int(pdf.groupby("bulan")["hujan_total"].mean().idxmin())
    rw = hasil["ringkas_wilayah"].iloc[0]
    return [
        f"Kota terbasah: <b>{basah['kota']}</b> ({basah['hujan_per_tahun']:,.0f} mm/tahun); terkering: <b>{kering['kota']}</b> ({kering['hujan_per_tahun']:,.0f} mm/tahun).",
        f"Kota terpanas (rata-rata): <b>{panas['kota']}</b> ({panas['suhu_rata']} \u00b0C); tersejuk: <b>{sejuk['kota']}</b> ({sejuk['suhu_rata']} \u00b0C).",
        f"Bulan terbasah secara keseluruhan: <b>{NAMA_BULAN[bulan_basah - 1]}</b>; terkering: <b>{NAMA_BULAN[bulan_kering - 1]}</b>.",
        f"Wilayah dengan hujan bulanan tertinggi: <b>{rw['wilayah']}</b> ({rw['hujan_bulanan']} mm/bulan).",
        f"Data: {len(pdf):,} baris bulanan dari {pdf['kota'].nunique()} kota, periode {MULAI} s.d. {AKHIR}.",
    ]

def tabel_html(df):
    return df.to_html(index=False, border=0, classes="t", float_format=lambda x: f"{x:,.2f}")

def bangun_laporan():
    bagian = [f"<h1>Atlas Iklim Kota-kota Indonesia - Kelompok {KELOMPOK}</h1>",
              f"<p>Dibuat {time.strftime('%Y-%m-%d %H:%M')} | Master: {socket.gethostname()} | RUN {RUN_ID}</p>",
              "<h2>Temuan utama</h2><ul>" + "".join(f"<li>{t}</li>" for t in temuan(pdf_bulanan, hasil_sql)) + "</ul>",
              "<h2>Pembagian tugas</h2>",
              "<p>" + "; ".join(f"{w}: {n} tugas" for w, n in Counter(w for t in range(JUMLAH_TUGAS) for w in pelaksana[t]).items()) + "</p>"]
    for nama, judul in [("peringkat_hujan", "Peringkat curah hujan"), ("ringkas_wilayah", "Ringkasan per wilayah"),
                        ("bulan_terbasah", "Bulan terbasah tiap kota"), ("anomali_hujan", "Anomali hujan (> 2x rata-rata)")]:
        bagian.append(f"<h2>{judul}</h2>" + tabel_html(hasil_sql[nama]))
    if "ringkas_bench" in globals():
        bagian.append("<h2>Hasil benchmark partisi x Worker</h2>" + tabel_html(ringkas_bench))
    bagian.append("<h2>Visualisasi</h2>")
    for p in sorted(LAPORAN_DIR.glob("*.png")):                       # semua gambar: kasus nyata, performa, benchmark
        bagian.append(f"<h3>{p.stem}</h3><img src='data:image/png;base64,{b64(p)}' style='max-width:100%;margin:8px 0'>")
    css = ("body{font-family:Arial,sans-serif;max-width:980px;margin:24px auto;padding:0 12px;color:#1a202c}"
           "h1{color:#1a365d}h2{color:#2b6cb0;border-bottom:2px solid #c05621;padding-bottom:4px}"
           ".t{border-collapse:collapse;font-size:13px}.t th{background:#2b6cb0;color:#fff;padding:4px 8px}.t td{padding:3px 8px;border-bottom:1px solid #ddd}")
    isi = f"<!doctype html><html><head><meta charset='utf-8'><title>Laporan {KELOMPOK}</title><style>{css}</style></head><body>{''.join(bagian)}</body></html>"
    p = LAPORAN_DIR / f"laporan_{KELOMPOK}.html"
    p.write_text(isi, encoding="utf-8")
    pdf_bulanan.to_csv(LAPORAN_DIR / "data_bulanan_gabungan.csv", index=False)
    return p

laporan = bangun_laporan()
print("Laporan akhir :", laporan.resolve())
print("Data gabungan :", (LAPORAN_DIR / "data_bulanan_gabungan.csv").resolve())
for t in temuan(pdf_bulanan, hasil_sql):
    print("-", html.unescape(t.replace("<b>", "").replace("</b>", "")))
display(FileLink(str(laporan)))

## Sel 17. Kirim ulang tugas / tutup Spark

In [ ]:
# ===== SEL 17: Kirim ulang tugas yang belum lengkap / reset =====
belum = [t for t in range(JUMLAH_TUGAS) if t not in lengkap()]
print("Belum lengkap:", belum)
# kirim_tugas([t for t in SEMUA_TUGAS if t["tugas"] in belum])    # buka komentar untuk kirim ulang
# spark.stop()                                                      # tutup Spark setelah selesai